# Prompt Engineering Evaluation

Collects 1,000 GPT-5.2 votes for each of three prompt variants on the same
1,000 image pairs, then analyses which prompt produces a vote distribution
most similar to humans.

**Three prompt variants:**
- Option A: few-shot, no draw normalisation instruction
- Option B: few-shot, with draw normalisation instruction
- Option C: zero-shot, with draw normalisation instruction

**Selection criterion:** Jensen-Shannon divergence from human vote distribution
on the same 1,000 pairs. Lower = more similar to human.

**Run cells in order. Section A collects votes. Section B analyses them.**

## 1. Install dependencies

In [ ]:
import sys, subprocess
for pkg in ['azure-storage-blob', 'openai', 'Pillow', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   check=True)
print('Done.')

## 2. Imports

In [ ]:
import os, io, re, time, base64, uuid, glob, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from PIL import Image
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient
from openai import AzureOpenAI
from scipy.stats import chi2_contingency
from scipy.spatial.distance import jensenshannon
warnings.filterwarnings('ignore')

BLUE   = '#2171B5'
ORANGE = '#E8743B'
GREEN  = '#1B7837'
GREY   = '#888888'
BG     = 'white'
PANEL  = '#F8F8F8'
plt.rcParams['font.family'] = 'DejaVu Sans'
print('Imports loaded.')

## 3. Configuration

Edit this cell before running.

In [ ]:
# ============================================================
# PATHS
# ============================================================
HUMAN_VOTES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data\human_votes_05_06_26.csv'
OUTPUT_DIR       = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\prompt_engineering'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
N_PAIRS        = 1000
RANDOM_SEED    = 42
RETRY_MAX      = 3
RATE_LIMIT_SEC = 0.5

# ============================================================
# AZURE OPENAI
# ============================================================
OPENAI_MODEL_ID       = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-5.2')
AZURE_OPENAI_ENDPOINT = os.getenv('DEPLOYMENT_NAME',
                                   'https://YOUR_AZURE_OPENAI_RESOURCE.openai.azure.com/')
AZURE_OPENAI_API_KEY  = os.getenv(
    'AZURE_OPENAI_API_KEY',
    'YOUR_AZURE_OPENAI_API_KEY'
)
client = AzureOpenAI(
    api_version    = os.getenv('OPENAI_API_VERSION', '2024-12-01-preview'),
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
)

# ============================================================
# AZURE BLOB
# ============================================================
AZURE_CONN_STR = os.getenv(
    'AZURE_CONN_STR',
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
BLOB_CONTAINER = 'london'
IMAGE_PREFIX   = 'training/training/Images/'

print('Configuration loaded.')
print(f'  Model   : {OPENAI_MODEL_ID}')
print(f'  Pairs   : {N_PAIRS:,} per prompt variant')
print(f'  Output  : {OUTPUT_DIR}')

## 4. Prompt variants

**Option A** — few-shot without draw normalisation instruction.
Tests whether the draw instruction is necessary to correct RLHF draw
suppression (Liu, Ge and Zhu, 2024; Ouyang et al., 2022).

**Option B** — few-shot with draw normalisation instruction.
Fixed reference prompt with draw instruction embedded inline
(Brown et al., 2020; He et al., 2024).

**Option C** — zero-shot with draw normalisation instruction.
Tests whether worked examples are necessary (Brown et al., 2020;
He et al., 2024) or whether the model performs equivalently without
them (Kojima et al., 2022).

In [ ]:
Option_A = (
    'You will see two images: Image A (left) and Image B (right). '
    'Which image looks safer for bicycle users? '
    'Choose 1 if Image A looks safer, 2 if Image B looks safer, '
    'or 0 if both look equally safe.\n'
    'Respond with ONE line only:\n'
    '[VOTE]: [Vote Number] - [reason]\n\n'
    'Examples:\n'
    '[VOTE]: 1 - Image A has a physically separated cycle track and '
    'lower motor traffic volume, reducing conflict risk.\n'
    '[VOTE]: 2 - Image B has a wider road with lower motor traffic '
    'speed and no close vehicle passing.\n'
    '[VOTE]: 0 - Both streets are quiet residential roads with '
    'similar traffic volume and no dedicated cycling provision.\n\n'
    'Now decide.'
)

Option_B = (
    'You will see two images: Image A (left) and Image B (right). '
    'Which image looks safer for bicycle users? '
    'Choose 1 if Image A looks safer, 2 if Image B looks safer, '
    'or 0 if both look equally safe - many pairs look similarly safe '
    'and if there is no clear meaningful difference, choose 0.\n'
    'Respond with ONE line only:\n'
    '[VOTE]: [Vote Number] - [reason]\n\n'
    'Examples:\n'
    '[VOTE]: 1 - Image A has a physically separated cycle track and '
    'lower motor traffic volume, reducing conflict risk.\n'
    '[VOTE]: 2 - Image B has a wider road with lower motor traffic '
    'speed and no close vehicle passing.\n'
    '[VOTE]: 0 - Both streets are quiet residential roads with '
    'similar traffic volume and no dedicated cycling provision.\n\n'
)

Option_C = (
    'You will see two images: Image A (left) and Image B (right). '
    'Which image looks safer for bicycle users? '
    'Choose 1 if Image A looks safer, 2 if Image B looks safer, '
    'or 0 if both look equally safe - many pairs look similarly safe '
    'and if there is no clear meaningful difference, choose 0.\n'
    'Respond with ONE line only:\n'
    '[VOTE]: [Vote Number] - [reason]\n\n'
    'Now decide.'
)

PROMPTS = {
    'Option_A': Option_A,
    'Option_B': Option_B,
    'Option_C': Option_C,
}

print('Prompt variants loaded.')
for name, prompt in PROMPTS.items():
    print(f'\n--- {name} ---')
    print(prompt)

## 5. Load human votes and sample 1,000 pairs

In [ ]:
human_df = pd.read_csv(HUMAN_VOTES_PATH)
for col in ['image_name1', 'image_name2']:
    if col in human_df.columns:
        human_df[col] = human_df[col].apply(
            lambda x: os.path.basename(str(x).strip())
        )
human_df = human_df[['image_name1','image_name2','vote']].dropna().copy()
human_df['vote'] = human_df['vote'].astype(int)
human_df['pair_key'] = human_df.apply(
    lambda r: tuple(sorted([r['image_name1'], r['image_name2']])), axis=1
)
unique_pairs = human_df.drop_duplicates(subset='pair_key').copy()
sample_pairs = unique_pairs.sample(
    n=N_PAIRS, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f'Human votes loaded      : {len(human_df):,}')
print(f'Unique pairs            : {len(unique_pairs):,}')
print(f'Pairs sampled           : {len(sample_pairs):,}')
print(f'Human draw rate (sample): {(sample_pairs["vote"]==0).mean()*100:.2f}%')
print(f'Human vote distribution : '
      f'{sample_pairs["vote"].value_counts().sort_index().to_dict()}')

## 6. Azure blob connection

In [ ]:
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

def download_blob_to_pil(blob_name, timeout=30):
    blob_data = container_client.download_blob(
        blob_name, timeout=timeout
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')

print('Testing Azure connection...')
test_blob = IMAGE_PREFIX + sample_pairs['image_name1'].iloc[0]
try:
    img_test = download_blob_to_pil(test_blob)
    print(f'  Connection OK — test image size: {img_test.size}')
except Exception as e:
    print(f'  Connection FAILED: {e}')
    raise

## 7. API call and vote parser

In [ ]:
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format='JPEG', quality=quality)
    return base64.b64encode(buff.getvalue()).decode('utf-8')


def call_gpt(prompt_text, pil1, pil2):
    uri1 = f'data:image/jpeg;base64,{pil_to_base64_jpeg(pil1)}'
    uri2 = f'data:image/jpeg;base64,{pil_to_base64_jpeg(pil2)}'
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text',      'text': 'Image A:'},
            {'type': 'image_url', 'image_url': {'url': uri1}},
            {'type': 'text',      'text': '\nImage B:'},
            {'type': 'image_url', 'image_url': {'url': uri2}},
            {'type': 'text',      'text': f'\n\n{prompt_text}'},
        ],
    }]
    response = client.chat.completions.create(
        model=OPENAI_MODEL_ID,
        messages=messages,
        max_completion_tokens=256,
    )
    return response.choices[0].message.content


def parse_vote(raw_txt):
    raw_clean = raw_txt.strip() if raw_txt else ''
    patterns  = [
        r'^\[VOTE\]\s*[:\-]\s*([0-2])\s*[:\-]\s*(.*)$',
        r'^([0-2])\s*[:\-]\s*(.*)$',
        r'([0-2])',
    ]
    for pat in patterns:
        m = re.search(pat, raw_clean, re.IGNORECASE | re.MULTILINE)
        if m:
            vote_num = int(m.group(1))
            reason   = m.group(2).strip() if len(m.groups()) >= 2 else ''
            return vote_num, reason, raw_clean
    return None, '', raw_clean


print('API client and parser ready.')

---
# Section A — Vote collection

Collects 1,000 votes for each of the three prompt variants.
All three prompts vote on the same 1,000 pairs.
Results are saved to CSV and stored in `all_results` for the analysis section.

In [ ]:
all_results    = {}
TIMESTAMP      = datetime.now().strftime('%Y%m%d_%H%M%S')

for prompt_name, prompt_text in PROMPTS.items():
    print(f'\n{"="*70}')
    print(f'Running {prompt_name} ({N_PAIRS:,} pairs)...')
    print(f'{"="*70}')

    results        = []
    parse_failures = []
    skipped        = []
    gpt_user_id    = str(uuid.uuid4())

    for i, row in tqdm(sample_pairs.iterrows(),
                       total=len(sample_pairs),
                       desc=prompt_name):
        name1 = row['image_name1']
        name2 = row['image_name2']

        pil1 = pil2 = None
        for attempt in range(RETRY_MAX):
            try:
                pil1 = download_blob_to_pil(IMAGE_PREFIX + name1)
                pil2 = download_blob_to_pil(IMAGE_PREFIX + name2)
                break
            except Exception as e:
                if attempt == RETRY_MAX - 1:
                    skipped.append({'prompt': prompt_name,
                                    'reason': f'download_failed: {str(e)[:80]}'})
                time.sleep(2 ** attempt)

        if pil1 is None or pil2 is None:
            continue

        raw_txt   = None
        vote_time = datetime.now().strftime('%d/%m/%Y %H:%M')
        for attempt in range(RETRY_MAX):
            try:
                raw_txt   = call_gpt(prompt_text, pil1, pil2)
                vote_time = datetime.now().strftime('%d/%m/%Y %H:%M')
                break
            except Exception as e:
                err_str = str(e)
                if 'content_policy_violation' in err_str or '400' in err_str:
                    skipped.append({'prompt': prompt_name,
                                    'reason': 'content_policy_violation'})
                    break
                if attempt == RETRY_MAX - 1:
                    skipped.append({'prompt': prompt_name,
                                    'reason': f'api_failed: {err_str[:80]}'})
                time.sleep(2 ** attempt)

        if raw_txt is None:
            continue

        ai_vote, reason, raw_clean = parse_vote(raw_txt)

        if ai_vote is None:
            parse_failures.append({'prompt': prompt_name, 'raw': raw_clean})
            skipped.append({'prompt': prompt_name,
                            'reason': f'parse_failed: {raw_clean[:80]}'})
            continue

        reason = reason.encode('utf-8', errors='replace').decode('utf-8')

        results.append({
            'prompt'      : prompt_name,
            'user_id'     : gpt_user_id,
            'image_name1' : IMAGE_PREFIX + name1,
            'image_name2' : IMAGE_PREFIX + name2,
            'vote'        : ai_vote,
            'location'    : '',
            'voting_time' : vote_time,
            'model_id'    : OPENAI_MODEL_ID,
            'reason'      : reason,
        })

        time.sleep(RATE_LIMIT_SEC)

    df_out   = pd.DataFrame(results)
    out_path = os.path.join(
        OUTPUT_DIR,
        f'pe_votes_{prompt_name}_{TIMESTAMP}.csv'
    )
    df_out.to_csv(out_path, index=False, encoding='utf-8-sig')
    all_results[prompt_name] = df_out

    draw_rate = (df_out['vote'] == 0).mean() * 100 if len(df_out) > 0 else 0
    print(f'  Processed      : {len(df_out):,}')
    print(f'  Parse failures : {len(parse_failures)}')
    print(f'  Skipped        : {len(skipped)}')
    print(f'  Draw rate      : {draw_rate:.2f}%')
    print(f'  Vote dist      : '
          f'{df_out["vote"].value_counts().sort_index().to_dict() if len(df_out)>0 else {}}')
    print(f'  Saved to       : {out_path}')

print('\nAll three prompt variants complete.')

# Criteria application: A: Vote distribution and B: Pair-level agreement

In [ ]:
RESULTS_DIR      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\prompt_engineering'
HUMAN_VOTES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data\human_votes_05_06_26.csv'
N_PAIRS          = 500
RANDOM_SEED      = 42

def load_latest(results_dir, option_name):
    pattern = os.path.join(results_dir, f'pe_votes_{option_name}_*.csv')
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No file found for {option_name}')
    path = files[-1]
    print(f'  {option_name}: {os.path.basename(path)} ({len(pd.read_csv(path)):,} rows)')
    return pd.read_csv(path)

print('Loading GPT vote files...')
all_results = {
    'Option_A': load_latest(RESULTS_DIR, 'Option_A'),
    'Option_B': load_latest(RESULTS_DIR, 'Option_B'),
    'Option_C': load_latest(RESULTS_DIR, 'Option_C'),
}

# normalise image names
for name, df in all_results.items():
    for col in ['image_name1', 'image_name2']:
        df[col] = df[col].apply(lambda x: os.path.basename(str(x).strip()))
    df['pair_key'] = df.apply(
        lambda r: tuple(sorted([r['image_name1'], r['image_name2']])), axis=1
    )
    all_results[name] = df

print('Done.')

In [ ]:
# load raw human votes
human_df = pd.read_csv(HUMAN_VOTES_PATH)
for col in ['image_name1', 'image_name2']:
    if col in human_df.columns:
        human_df[col] = human_df[col].apply(
            lambda x: os.path.basename(str(x).strip())
        )
human_df = human_df[['user_id','image_name1','image_name2',
                      'vote','voting_time']].dropna().copy()
human_df['vote']        = human_df['vote'].astype(int)
human_df['voting_time'] = pd.to_datetime(
    human_df['voting_time'], format='mixed', dayfirst=True
)

# ── four behaviour filters ────────────────────────────────────────────────────
MIN_VOTES_SPEED    = 5
SPEED_THRESHOLD    = 2.0
MIN_VOTES_PATTERN  = 10
MAX_STREAK         = 10
ALT_RATE_THRESHOLD = 0.90

def compute_user_stats(df):
    counts     = df.groupby('user_id').size().rename('vote_count')
    vote_props = (
        df.groupby('user_id')['vote']
          .value_counts(normalize=True)
          .rename('prop').reset_index()
    )
    max_prop = vote_props.groupby('user_id')['prop'].max().rename('max_option_prop')
    df_s     = df.sort_values(['user_id','voting_time']).copy()
    df_s['dt'] = df_s.groupby('user_id')['voting_time'].diff().dt.total_seconds()
    mean_dt  = (
        df_s.groupby('user_id')['dt']
        .mean().replace({np.nan: np.inf})
        .rename('mean_dt_seconds')
    )
    return pd.concat([counts, max_prop, mean_dt], axis=1)

def compute_bias_threshold(vote_count):
    if vote_count > 100:  return 0.70
    elif vote_count > 50: return 0.75
    elif vote_count > 10: return 0.80
    else:                 return np.nan

def compute_pattern_stats(df):
    df_s    = df.sort_values(['user_id','voting_time']).copy()
    results = []
    for user_id, grp in df_s.groupby('user_id'):
        votes = grp['vote'].tolist()
        n     = len(votes)
        max_streak = current_streak = 1
        for i in range(1, n):
            if votes[i] == votes[i-1]:
                current_streak += 1
                max_streak = max(max_streak, current_streak)
            else:
                current_streak = 1
        transition_rate = (
            sum(votes[i] != votes[i-1] for i in range(1, n)) / (n-1)
            if n >= 2 else np.nan
        )
        results.append({'user_id': user_id, 'max_streak': max_streak,
                        'transition_rate': transition_rate})
    return pd.DataFrame(results).set_index('user_id')

# F1
us     = compute_user_stats(human_df)
us['bias_threshold'] = us['vote_count'].apply(compute_bias_threshold)
bad_f1 = us.index[~us['bias_threshold'].isna() &
                  (us['max_option_prop'] >= us['bias_threshold'])].tolist()
df_f1  = human_df[~human_df['user_id'].isin(bad_f1)].copy()

# F2
us_f1  = compute_user_stats(df_f1)
bad_f2 = us_f1.index[(us_f1['vote_count'] >= MIN_VOTES_SPEED) &
                      (us_f1['mean_dt_seconds'] < SPEED_THRESHOLD)].tolist()
df_f2  = df_f1[~df_f1['user_id'].isin(bad_f2)].copy()

# F3
us_f2  = compute_user_stats(df_f2).join(compute_pattern_stats(df_f2))
bad_f3 = us_f2.index[(us_f2['vote_count'] >= MIN_VOTES_PATTERN) &
                      (us_f2['max_streak'] >= MAX_STREAK)].tolist()
df_f3  = df_f2[~df_f2['user_id'].isin(bad_f3)].copy()

# F4
us_f3  = compute_user_stats(df_f3).join(compute_pattern_stats(df_f3))
bad_f4 = us_f3.index[(us_f3['vote_count'] >= MIN_VOTES_PATTERN) &
                      (us_f3['transition_rate'] >= ALT_RATE_THRESHOLD)].tolist()
filtered_human_df = df_f3[~df_f3['user_id'].isin(bad_f4)].copy()

print(f'Raw votes          : {len(human_df):,}')
print(f'After filtering    : {len(filtered_human_df):,} votes  '
      f'{filtered_human_df["user_id"].nunique():,} users')

# ── build pair lookup from filtered human votes ───────────────────────────────
filtered_human_df['pair_key'] = filtered_human_df.apply(
    lambda r: tuple(sorted([r['image_name1'], r['image_name2']])), axis=1
)
unique_human = filtered_human_df.drop_duplicates(subset='pair_key').copy()

# ── intersect with GPT voted pairs ────────────────────────────────────────────
# only keep pairs where both human (filtered) and all three GPT prompts voted
gpt_keys = set(all_results['Option_A']['pair_key']) & \
           set(all_results['Option_B']['pair_key']) & \
           set(all_results['Option_C']['pair_key'])
human_keys = set(unique_human['pair_key'])
overlap_keys = human_keys & gpt_keys

print(f'Unique filtered human pairs : {len(unique_human):,}')
print(f'GPT voted pairs (all 3)     : {len(gpt_keys):,}')
print(f'Overlapping pairs           : {len(overlap_keys):,}')

# sample 500 from overlap
overlap_df   = unique_human[unique_human['pair_key'].isin(overlap_keys)].copy()
sample_pairs = overlap_df.sample(
    n=min(N_PAIRS, len(overlap_df)), random_state=RANDOM_SEED
).reset_index(drop=True)

print(f'\nSample pairs                : {len(sample_pairs):,}')
print(f'Human draw rate (filtered)  : {(sample_pairs["vote"]==0).mean()*100:.2f}%')
print(f'Vote distribution           : '
      f'{sample_pairs["vote"].value_counts().sort_index().to_dict()}')

In [ ]:
# ── helper functions ──────────────────────────────────────────────────────────

def vote_distribution(df, vote_col='vote'):
    vc = df[vote_col].value_counts(normalize=True).reindex([0,1,2], fill_value=0)
    return vc.values

def build_pair_lookup(df):
    lookup = {}
    for _, row in df.iterrows():
        key = tuple(sorted([
            os.path.basename(str(row['image_name1']).strip()),
            os.path.basename(str(row['image_name2']).strip())
        ]))
        lookup[key] = int(row['vote'])
    return lookup

def pair_level_agreement(gpt_df, human_df):
    gpt_lookup   = build_pair_lookup(gpt_df)
    human_lookup = build_pair_lookup(human_df)
    overlap      = set(gpt_lookup.keys()) & set(human_lookup.keys())
    y_true = [human_lookup[k] for k in overlap]
    y_pred = [gpt_lookup[k]   for k in overlap]
    return np.array(y_true), np.array(y_pred), len(overlap)

# ── compute metrics ───────────────────────────────────────────────────────────
human_dist = vote_distribution(sample_pairs)
human_draw = (sample_pairs['vote'] == 0).mean() * 100

variants = {
    'Option A': all_results['Option_A'],
    'Option B': all_results['Option_B'],
    'Option C': all_results['Option_C'],
}

metrics       = {}
agree_metrics = {}

for name, df in variants.items():
    # restrict GPT votes to the 500 sampled pairs
    gpt_500 = df[df['pair_key'].isin(set(sample_pairs['pair_key']))].copy()

    # criterion A: distribution
    dist      = vote_distribution(gpt_500)
    draw_rate = (gpt_500['vote'] == 0).mean() * 100
    draw_diff = abs(draw_rate - human_draw)
    js_div    = float(jensenshannon(dist, human_dist, base=2))
    gpt_c     = gpt_500['vote'].value_counts().reindex([0,1,2], fill_value=0)
    hum_c     = sample_pairs['vote'].value_counts().reindex([0,1,2], fill_value=0)
    chi2, p, _, _ = chi2_contingency(np.array([gpt_c.values, hum_c.values]))
    metrics[name] = {
        'dist': dist, 'draw_rate': draw_rate,
        'draw_diff': draw_diff, 'js_div': js_div,
        'chi2': chi2, 'p': p, 'n': len(gpt_500)
    }

    # criterion B: pair-level agreement
    y_true, y_pred, n_ov = pair_level_agreement(gpt_500, sample_pairs)
    if n_ov > 0:
        recall_c = recall_score(
            y_true, y_pred, labels=[0,1,2], average=None, zero_division=0
        )
        agree_metrics[name] = {
            'n_overlap'   : n_ov,
            'accuracy'    : accuracy_score(y_true, y_pred),
            'bal_acc'     : balanced_accuracy_score(y_true, y_pred),
            'kappa'       : cohen_kappa_score(y_true, y_pred),
            'mcc'         : matthews_corrcoef(y_true, y_pred),
            'recall_draw' : recall_c[0],
            'recall_a'    : recall_c[1],
            'recall_b'    : recall_c[2],
        }

rows         = list(metrics.values())
agree_rows   = list(agree_metrics.values())
labels_short = list(metrics.keys())

# ── print criterion A ─────────────────────────────────────────────────────────
print('CRITERION A — Vote distribution similarity')
print(f'Human distribution (filtered, {len(sample_pairs):,} pairs): {human_dist.round(3)}')
print(f'Human draw rate: {human_draw:.2f}%')
print()
print(f'{"Metric":<38} {"Option A":>12} {"Option B":>12} {"Option C":>12}')
print('-' * 78)
for label, row in zip(labels_short, rows):
    print(f'  {label}: n={row["n"]}  dist={row["dist"].round(3)}')
print(f'\n{"Draw rate (%)":<38}', end='')
for r in rows: print(f' {r["draw_rate"]:>11.2f}%', end='')
print(f'\n{"Draw rate diff from human (pp)":<38}', end='')
for r in rows: print(f' {r["draw_diff"]:>11.2f}pp', end='')
print(f'\n{"JS divergence (lower = better)":<38}', end='')
for r in rows: print(f' {r["js_div"]:>12.4f}', end='')
print(f'\n{"Chi-squared p-value":<38}', end='')
for r in rows: print(f' {r["p"]:>12.4f}', end='')
print()

# ── print criterion B ─────────────────────────────────────────────────────────
print()
print('CRITERION B — Pair-level agreement with filtered human votes')
print(f'{"Metric":<38} {"Option A":>12} {"Option B":>12} {"Option C":>12}')
print('-' * 78)
print(f'{"Overlapping pairs":<38}', end='')
for r in agree_rows: print(f' {r["n_overlap"]:>12,}', end='')
print(f'\n{"Accuracy":<38}', end='')
for r in agree_rows: print(f' {r["accuracy"]:>12.4f}', end='')
print(f'\n{"Balanced accuracy":<38}', end='')
for r in agree_rows: print(f' {r["bal_acc"]:>12.4f}', end='')
print(f'\n{"Cohen kappa":<38}', end='')
for r in agree_rows: print(f' {r["kappa"]:>12.4f}', end='')
print(f'\n{"Matthews corrcoef":<38}', end='')
for r in agree_rows: print(f' {r["mcc"]:>12.4f}', end='')
print(f'\n{"Recall — Equal (draw)":<38}', end='')
for r in agree_rows: print(f' {r["recall_draw"]:>12.4f}', end='')
print(f'\n{"Recall — Image A safer":<38}', end='')
for r in agree_rows: print(f' {r["recall_a"]:>12.4f}', end='')
print(f'\n{"Recall — Image B safer":<38}', end='')
for r in agree_rows: print(f' {r["recall_b"]:>12.4f}', end='')
print()

best_js = labels_short[np.argmin([r['js_div']  for r in rows])]
best_ba = labels_short[np.argmax([r['bal_acc'] for r in agree_rows])]
best_k  = labels_short[np.argmax([r['kappa']   for r in agree_rows])]
print(f'Best by JS divergence     : {best_js}')
print(f'Best by balanced accuracy : {best_ba}')
print(f'Best by Cohen kappa       : {best_k}')

In [ ]:
variant_colors = [GREEN, BLUE, ORANGE]
vote_labels    = ['Equal (0)', 'Image A safer (1)', 'Image B safer (2)']
x              = np.arange(3)
width          = 0.18

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor(BG)
fig.suptitle(
    'Prompt engineering evaluation — two-criterion analysis\n'
    f'Reference: filtered human votes ({len(sample_pairs):,} pairs)',
    fontsize=12, fontweight='bold', y=1.01
)

# ── row 1: criterion A ────────────────────────────────────────────────────────

# top-left: vote distribution
ax = axes[0, 0]
ax.set_facecolor(PANEL)
bars = ax.bar(x - 1.5*width, human_dist * 100, width,
              color=GREY, alpha=0.85, edgecolor='white', label='Human (filtered)')
for bar, val in zip(bars, human_dist * 100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', fontsize=7)
for i, (name, color) in enumerate(zip(labels_short, variant_colors)):
    df_500 = all_results[name.replace(' ','_')][
        all_results[name.replace(' ','_')]['pair_key'].isin(
            set(sample_pairs['pair_key'])
        )
    ]
    dist   = vote_distribution(df_500) * 100
    offset = (i - 0.5) * width
    bars   = ax.bar(x + offset, dist, width,
                    color=color, alpha=0.85, edgecolor='white', label=name)
    for bar, val in zip(bars, dist):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.1f}%', ha='center', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(vote_labels, fontsize=8)
ax.set_ylabel('Share of votes (%)', fontsize=9)
ax.set_title('Criterion A: Vote distribution', fontsize=10, fontweight='bold')
ax.legend(fontsize=7.5, frameon=True, framealpha=0.9)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.2, linestyle='--')

# top-middle: draw rate
ax2 = axes[0, 1]
ax2.set_facecolor(PANEL)
all_labels     = ['Human\n(filtered)'] + labels_short
all_draw_rates = [human_draw] + [r['draw_rate'] for r in rows]
all_colors     = [GREY] + variant_colors
bars = ax2.bar(all_labels, all_draw_rates,
               color=all_colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, all_draw_rates):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax2.axhline(human_draw, color='#c0392b', linewidth=1.5, linestyle='--',
            alpha=0.8, label=f'Human ({human_draw:.1f}%)')
ax2.set_ylabel('Draw rate (%)', fontsize=9)
ax2.set_title('Criterion A: Draw rate vs human', fontsize=10, fontweight='bold')
ax2.legend(fontsize=8, frameon=True, framealpha=0.9)
ax2.spines[['top','right']].set_visible(False)
ax2.grid(axis='y', alpha=0.2, linestyle='--')
ax2.tick_params(axis='x', labelsize=8)

# top-right: JS divergence
ax3 = axes[0, 2]
ax3.set_facecolor(PANEL)
js_vals = [r['js_div'] for r in rows]
bars    = ax3.bar(labels_short, js_vals,
                  color=variant_colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, js_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
             f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax3.set_ylabel('JS divergence (lower = better)', fontsize=9)
ax3.set_title('Criterion A: JS divergence\nfrom human distribution',
              fontsize=10, fontweight='bold')
ax3.spines[['top','right']].set_visible(False)
ax3.grid(axis='y', alpha=0.2, linestyle='--')
ax3.tick_params(axis='x', labelsize=8)

# ── row 2: criterion B ────────────────────────────────────────────────────────

# bottom-left: balanced accuracy and kappa
ax4 = axes[1, 0]
ax4.set_facecolor(PANEL)
x2      = np.arange(len(labels_short))
w       = 0.3
ba_vals = [r['bal_acc'] for r in agree_rows]
k_vals  = [r['kappa']   for r in agree_rows]
bars1   = ax4.bar(x2 - w/2, ba_vals, w, color=variant_colors,
                  alpha=0.85, edgecolor='white', label='Balanced accuracy')
bars2   = ax4.bar(x2 + w/2, k_vals,  w, color=variant_colors,
                  alpha=0.5,  edgecolor='white', hatch='//', label='Cohen kappa')
for bar, val in zip(bars1, ba_vals):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
for bar, val in zip(bars2, k_vals):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{val:.3f}', ha='center', fontsize=8)
ax4.set_xticks(x2)
ax4.set_xticklabels(labels_short, fontsize=9)
ax4.set_ylabel('Score', fontsize=9)
ax4.set_title('Criterion B: Balanced accuracy\nand Cohen kappa vs human',
              fontsize=10, fontweight='bold')
ax4.legend(fontsize=8, frameon=True, framealpha=0.9)
ax4.spines[['top','right']].set_visible(False)
ax4.grid(axis='y', alpha=0.2, linestyle='--')

# bottom-middle: per-class recall
ax5 = axes[1, 1]
ax5.set_facecolor(PANEL)
x3     = np.arange(len(labels_short))
w2     = 0.22
rd     = [r['recall_draw'] for r in agree_rows]
ra     = [r['recall_a']    for r in agree_rows]
rb     = [r['recall_b']    for r in agree_rows]
bars_d = ax5.bar(x3 - w2, rd, w2, color=GREY,   alpha=0.85,
                 edgecolor='white', label='Recall — Equal (0)')
bars_a = ax5.bar(x3,      ra, w2, color=BLUE,   alpha=0.85,
                 edgecolor='white', label='Recall — A safer (1)')
bars_b = ax5.bar(x3 + w2, rb, w2, color=ORANGE, alpha=0.85,
                 edgecolor='white', label='Recall — B safer (2)')
for bars_set, vals in [(bars_d, rd), (bars_a, ra), (bars_b, rb)]:
    for bar, val in zip(bars_set, vals):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.2f}', ha='center', fontsize=7.5)
ax5.set_xticks(x3)
ax5.set_xticklabels(labels_short, fontsize=9)
ax5.set_ylabel('Recall', fontsize=9)
ax5.set_ylim(0, 1.1)
ax5.set_title('Criterion B: Per-class recall\nvs human majority vote',
              fontsize=10, fontweight='bold')
ax5.legend(fontsize=7.5, frameon=True, framealpha=0.9)
ax5.spines[['top','right']].set_visible(False)
ax5.grid(axis='y', alpha=0.2, linestyle='--')

# bottom-right: combined ranking
ax6 = axes[1, 2]
ax6.set_facecolor(PANEL)
js_ranks  = pd.Series([r['js_div']  for r in rows]).rank().values
ba_ranks  = pd.Series([r['bal_acc'] for r in agree_rows]).rank(ascending=False).values
k_ranks   = pd.Series([r['kappa']   for r in agree_rows]).rank(ascending=False).values
avg_ranks = (js_ranks + ba_ranks + k_ranks) / 3
bars = ax6.bar(labels_short, avg_ranks,
               color=variant_colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, avg_ranks):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
ax6.set_ylabel('Average rank (lower = better)', fontsize=9)
ax6.set_title('Combined ranking\n(JS div + balanced accuracy + kappa)',
              fontsize=10, fontweight='bold')
ax6.set_ylim(0, 4)
ax6.spines[['top','right']].set_visible(False)
ax6.grid(axis='y', alpha=0.2, linestyle='--')
ax6.tick_params(axis='x', labelsize=9)

plt.tight_layout(h_pad=3.0)
plt.show()

In [ ]:
best_overall = labels_short[np.argmin(avg_ranks)]

print('=' * 65)
print('RECOMMENDATION')
print('=' * 65)
print(f'Reference: filtered human votes ({len(sample_pairs):,} pairs)')
print()
print('Criterion A — Vote distribution similarity:')
print(f'  {"Prompt":<12} {"JS div":>10} {"Draw diff":>12} {"Distinguishable?"}')
print('  ' + '-' * 55)
for label, row in zip(labels_short, rows):
    sig = 'No  (p > 0.05)' if row['p'] > 0.05 else 'Yes (p <= 0.05)'
    print(f'  {label:<12} {row["js_div"]:>10.4f} '
          f'{row["draw_diff"]:>10.2f}pp   {sig}')
print()
print('Criterion B — Pair-level agreement:')
print(f'  {"Prompt":<12} {"Bal. acc":>10} {"Kappa":>8} '
      f'{"Draw recall":>13} {"A recall":>10} {"B recall":>10}')
print('  ' + '-' * 65)
for label, row in zip(labels_short, agree_rows):
    print(f'  {label:<12} {row["bal_acc"]:>10.4f} {row["kappa"]:>8.4f} '
          f'{row["recall_draw"]:>13.4f} {row["recall_a"]:>10.4f} '
          f'{row["recall_b"]:>10.4f}')
print()
print('Combined ranking (JS divergence + balanced accuracy + kappa):')
for label, rank in zip(labels_short, avg_ranks):
    print(f'  {label:<12} avg rank = {rank:.2f}')
print()
print(f'Recommended prompt : {best_overall}')
print()
print('Note: draw recall is expected to be near-zero for all prompts.')
print('GPT suppresses draws due to RLHF binary training regardless of')
print('prompt design. JS divergence and balanced accuracy are the')
print('primary criteria for selecting the final prompt.')